In [0]:
%pip install -qq langchain-text-splitters

Note: you may need to restart the kernel using %restart_python or dbutils.library.restartPython() to use updated packages.


In [0]:
parsed_table = "dmi_genomics_qa_team_consenting_practice.bg_genai.docs_parsed"
chunked_table = "dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked"

parsed_df = spark.read.table(parsed_table)

print(f"Loaded parsed documents from: {parsed_table}")
parsed_df.printSchema()

Loaded parsed documents from: dmi_genomics_qa_team_consenting_practice.bg_genai.docs_parsed
root
 |-- path: string (nullable = true)
 |-- modificationTime: timestamp (nullable = true)
 |-- length: long (nullable = true)
 |-- parsed_content: variant (nullable = true)



In [0]:
from pyspark.sql.functions import expr

# Choose a Databricks foundation model (or your own serving endpoint name)
ENDPOINT = "databricks-gpt-oss-20b"

# Example prompt for the LLM
prompt_prefix = '''
You are a helpful assistant. Given a JSON object representing a parsed document (with pages, elements, and metadata), convert the content into clean, readable markdown. Use "== page ==" to separate each page. Preserve important structure such as headers, tables, and captions. Do not include any JSON or code blocks in the output—just the clean markdown text.

JSON:

'''

# Apply ai_query to batch process the parsed JSON text
transformed_df = (
    parsed_df.withColumn(
        "clean_markdown_text",
        expr(f"""
          ai_query(
            '{ENDPOINT}',
            CONCAT('{prompt_prefix}', CAST(parsed_content AS STRING)),
            responseFormat => '{{"type":"text"}}'
          )
        """)
    )
)

display(transformed_df.select("path", "clean_markdown_text"))

path clean_markdown_text dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf == page ==

# Spark Adaptive Query Execution (AQE) Internals

## Prepared By: Kaviraj T U

## How Spark Rewrites Your Plan at Runtime

Many engineers tune Spark jobs manually:

- Adjust shuffle partitions 
- Change join strategy 
- Handle skew manually 
- Increase executor memory 

But modern Spark can do something powerful: 
It can change the execution plan at runtime. 
That feature is called Adaptive Query Execution (AQE). 
And understanding its internals is a senior‑level Spark skill.

## What is AQE?

Normally, Spark:

- Builds a logical plan 
- Optimizes it 
- Creates a physical plan 
- Executes it 

That plan is fixed before execution starts.

== page ==

# Problem?

Spark doesn't know the real/data distribution until shuffle happens. 
AQE solves this. 
It collects runtime statistics and modifies the physical plan dynamically.

## When Does AQE Kick In?

AQE activates after a shuffle stage completes. 
At that moment Spark now knows:

- Actual partition sizes 
- Real data distribution 
- True join sizes 
- Skewed partitions 

Using this information, it can re‑optimize the next stage.

## What AQE Actually Optimizes

### Dynamic Shuffle Partition Coalescing

Default shuffle partitions: 
`spark.sql.shuffle.partitions=200` 

But what if data is small?

Without AQE: 
→ 200 tiny partitions 

Too many small tasks 
Scheduling overhead 

### With AQE:

→ Spark merges small partitions automatically 
Fewer, optimal‑sized tasks 

Result: 

- Less overhead 
- Better resource usage 

### Dynamic Join Strategy Switching

At planning time, Spark may choose: 

- Sort‑Merge Join. 

But at runtime it may discover: 

- One side is actually small enough to broadcast. 

AQE can switch: 

- Sort‑Merge → Broadcast Join 

Without you changing code. 
This can reduce shuffle dramatically.

### Skew Join Optimization

== page ==

If one partition is much larger:

### Without AQE:

- One executor overloaded 
→ Long‑running task 
Potential OOM 

With AQE:

- Spark splits skewed partition 
- Processes it in parallel 
- Reduces imbalance 

This is huge for production workloads.

## How to Enable AQE

`spark.conf.set("spark.sql.adaptive.enabled", "true")`

Other useful configs: 

`spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")` 
`spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")`

In Spark 3+, AQE is often enabled by default.

== page ==

# How to Verify AQE is Working

In Spark UI: 

- Look at SQL tab 
- Check "AdaptiveSparkPlan" in physical plan 
- Compare initial plan vs final plan 

You'll see plan changes during execution. 
That's AQE in action.

## When AQE Doesn't Help

AQE is powerful — but not magic. 
It won't fix: 

- Poor partitioning strategy 
- Massive data skew beyond threshold 
- Bad UDF design 
- Memory misconfiguration 

It optimizes around runtime statistics — not architectural flaws.

## Production Insight

== page ==

# Before AQE:

Engineers manually tuned everything.

# After AQE:

- Spark handles many optimizations automatically.

# But...Senior engineers still:

- Inspect execution plans 
- Monitor skew behavior 
- Validate join strategy 
- Tune thresholds carefully 

## Automation helps.

Understanding still wins.

# Final Takeaway

AQE makes Spark adaptive. 
But if you don't understand: 

- How shuffle works 
- How joins are selected 
- How partition sizing affects execution 

You won't know whether AQE helped — or hid a deeper issue. 
Distributed systems are not about writing queries. 
They're about understanding execution.

== page ==

AQE is Spark's way of becoming smarter at runtime. 
Your job is to be smarter than the runtime. 
Thank You.! 
Happy Learning... dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE_Catalystoptimizer.pdf == page 0 ==

Catalyst Optimizer vs AQE

### 1 Execution Time

Catalyst Optimizer → Works before execution (compile time)

In [0]:
import json
from typing import Any, Optional

def _page_id_from_bbox(bbox: Any) -> Optional[int]:
    """
    bbox can be a list[dict], a dict, or None. Return bbox.page_id if present.
    Kept intentionally simple.
    """
    if not bbox:
        return None
    if isinstance(bbox, list) and bbox:
        first = bbox[0] or {}
        return first.get("page_id")
    if isinstance(bbox, dict):
        return bbox.get("page_id")
    return None

def extract_contents_from_json(json_str: str) -> str:
    """
    - Concatenate element 'content' (fallback to 'description' if missing).
    - Insert '== page ==' when page_id changes.
    - Insert an extra newline after elements whose type != 'text'.
    - Return an error string on failure (for easy debugging in DataFrame).
    """
    try:
        doc = json.loads(json_str) if isinstance(json_str, str) else json_str
        if not isinstance(doc, dict):
            return ""

        # Support both {"document":{"elements":[...]}} and {"elements":[...]}
        document = doc.get("document", doc)
        elements = document.get("elements", []) if isinstance(document, dict) else []
        if not isinstance(elements, list):
            return ""

        out_lines = []
        current_page = None

        for el in elements:
            if not isinstance(el, dict):
                continue

            # Page divider on change
            pid = _page_id_from_bbox(el.get("bbox"))
            if pid is not None and current_page is not None and pid != current_page:
                out_lines.append("")
                out_lines.append("== page ==")
                out_lines.append("")
            if pid is not None:
                current_page = pid

            # Content (fallback to description)
            c = el.get("content")
            if not (isinstance(c, str) and c.strip()):
                c = el.get("description")
            if isinstance(c, str) and c.strip():
                out_lines.append(c)

                # Extra newline after non-text elements
                t = (el.get("type") or "").lower()
                if t != "text":
                    out_lines.append("")  # produces a blank line after joining

        return "\n".join(out_lines)

    except Exception as e:
        return f"Error: {str(e)}"


# A small factory to create a PySpark UDF that uses the function above.
def extract_contents_udf():
    from pyspark.sql.types import StringType
    from pyspark.sql.functions import udf
    @udf(StringType())
    def _udf(json_str):
        try:
            return extract_contents_from_json(json_str)
        except Exception as e:
            return f"Error: {str(e)}"
    return _udf

In [0]:
from pyspark.sql import functions as F

# Convert VARIANT/struct/map to a JSON string first (avoids VariantVal issues)
safe_json_col = F.coalesce(
    F.to_json(F.col("parsed_content")),
    F.col("parsed_content").cast("string")
)

# Apply the UDF
plain_text_df = parsed_df.withColumn(
    "plain_text",
    extract_contents_udf()(safe_json_col)
)

display(plain_text_df.select("path", "plain_text"))

path plain_text dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf Spark Adaptive Query
Execution (AQE) Internals

Prepared By: Kaviraj T U

How Spark Rewrites Your Plan at Runtime

Many engineers tune Spark jobs manually:
Adjust shuffle partitions
Change join strategy
Handle skew manually
Increase executor memory
But modern Spark can do something powerful:
It can change the execution plan at runtime.
That feature is called Adaptive Query Execution (AQE).
And understanding its internals is a senior-level Spark skill.
What is AQE?

Normally, Spark:
Builds a logical plan
2 Optimizes it
Creates a physical plan
Executes it
That plan is fixed before execution starts.
Spark Adaptive Query Execution (AQE) Internals


== page ==

Problem?

Spark doesn't know the real/data distribution until shuffle happens.
AQE solves this.
It collects runtime statistics and modifies the physical plan dynamically.
When Does AQE Kick In?

AQE activates after a shuffle stage completes.
At that moment Spark now knows:
Actual partition sizes
Real data distribution
True join sizes
Skewed partitions
Using this information, it can re-optimize the next stage.
What AQE Actually Optimizes

Dynamic Shuffle Partition Coalescing

Default shuffle partitions:
spark.sql.shuffle.partitions=200
But what if data is small?
Without AQE:
→ 200 tiny partitions
Spark Adaptive Query Execution (AQE) Internals

2


== page ==

Too many small tasks
Scheduling overhead
With AQE:

→ Spark merges small partitions automatically
Fewer, optimal-sized tasks
Result:

Less overhead
Better resource usage
2 Dynamic Join Strategy Switching

At planning time, Spark may choose:
Sort-Merge Join.
But at runtime it may discover:
One side is actually small enough to broadcast.
AQE can switch:
Sort-Merge → Broadcast Join

Without you changing code.
This can reduce shuffle dramatically.
Skew Join Optimization

Spark Adaptive Query Execution (AQE) Internals

3


== page ==

If one partition is much larger:
Without AQE:
One executor overloaded
→ Long-running task
Potential OOM
With AQE:
→ Spark splits skewed partition
→ Processes it in parallel
→ Reduces imbalance
This is huge for production workloads.
How to Enable AQE

spark.conf.set("spark.sql.adaptive.enabled", "true")
Other useful configs:
spark.conf.set("spark.sql.adaptive.coalescePartitions.enabled", "true")
spark.conf.set("spark.sql.adaptive.skewJoin.enabled", "true")
In Spark 3+, AQE is often enabled by default.
Spark Adaptive Query Execution (AQE) Internals


== page ==

How to Verify AQE is Working

In Spark UI:
Look at SQL tab
Check "AdaptiveSparkPlan" in physical plan
Compare initial plan vs final plan
You'll see plan changes during execution.
That's AQE in action.
When AQE Doesn't Help

AQE is powerful — but not magic.
It won't fix:
Poor partitioning strategy
Massive data skew beyond threshold
Bad UDF design
Memory misconfiguration
It optimizes around runtime statistics — not architectural flaws.
Production Insight

Spark Adaptive Query Execution (AQE) Internals

5


== page ==

Before AQE:

Engineers manually tuned everything.
After AQE:

- Spark handles many optimizations automatically.
But...Senior engineers still:

Inspect execution plans
Monitor skew behavior
Validate join strategy
Tune thresholds carefully
Automation helps.

Understanding still wins.
Final Takeaway

AQE makes Spark adaptive.
But if you don't understand:
How shuffle works
How joins are selected
How partition sizing affects execution
You won't know whether AQE helped — or hid a deeper issue.
Distributed systems are not about writing queries.
They're about understanding execution.
Spark Adaptive Query Execution (AQE) Internals

6


== page ==

AQE is Spark's way of becoming smarter at runtime.
Your job is to be smarter than the runtime.
Thank You.!
Happy Learning...
Spark Adaptive Query Execution (AQE) Internals
 dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE_Catalystoptimizer.pdf Catalyst Optimizer 

In [0]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from pyspark.sql.types import StructType, StructField, StringType
import pandas as pd

# Set chunking parameters: chunk_size controls the max length of each chunk, and chunk_overlap allows for overlapping text between chunks to improve retrieval quality.
CHUNK_SIZE = 2000
CHUNK_OVERLAP = 200

# Build the text splitter with preferred separators.
# This splitter will break the text at page markers or other natural boundaries, preserving document structure where possible.
splitter = RecursiveCharacterTextSplitter(
    chunk_size=CHUNK_SIZE,
    chunk_overlap=CHUNK_OVERLAP,
    separators=["\n== page ==\n", "== page ==", "\n\n", "\n", " ", ""]
)

# Define the output schema for the chunked DataFrame.
# Each row will contain the document path and a single text chunk.
schema = StructType([
    StructField("path", StringType(), True),
    StructField("chunk", StringType(), True),
])

def split_rows(iterator):
    """
    mapInPandas function: input pdfs with columns [path, plain_text],
    output rows [path, chunk].
    This function splits each document's text into chunks and yields them for DataFrame construction.
    """
    for pdf in iterator:
        out = []
        for _, row in pdf.iterrows():
            path = row["path"]
            text = row["plain_text"]
            if isinstance(text, str) and text.strip():
                for c in splitter.split_text(text):
                    if c and c.strip():
                        out.append((path, c))
        yield pd.DataFrame(out, columns=["path", "chunk"])

# Apply the splitter to the plain text DataFrame.
# This step transforms each document into multiple chunked rows for efficient downstream retrieval and embedding.
df_chunks = (
    plain_text_df
    .select("path", "plain_text")
    .mapInPandas(split_rows, schema=schema)
)

# Display the resulting chunked DataFrame for inspection.
display(df_chunks)
     

path,chunk
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"Spark Adaptive Query Execution (AQE) Internals Prepared By: Kaviraj T U How Spark Rewrites Your Plan at Runtime Many engineers tune Spark jobs manually: Adjust shuffle partitions Change join strategy Handle skew manually Increase executor memory But modern Spark can do something powerful: It can change the execution plan at runtime. That feature is called Adaptive Query Execution (AQE). And understanding its internals is a senior-level Spark skill. What is AQE? Normally, Spark: Builds a logical plan 2 Optimizes it Creates a physical plan Executes it That plan is fixed before execution starts. Spark Adaptive Query Execution (AQE) Internals == page == Problem? Spark doesn't know the real/data distribution until shuffle happens. AQE solves this. It collects runtime statistics and modifies the physical plan dynamically. When Does AQE Kick In? AQE activates after a shuffle stage completes. At that moment Spark now knows: Actual partition sizes Real data distribution True join sizes Skewed partitions Using this information, it can re-optimize the next stage. What AQE Actually Optimizes Dynamic Shuffle Partition Coalescing Default shuffle partitions: spark.sql.shuffle.partitions=200 But what if data is small? Without AQE: → 200 tiny partitions Spark Adaptive Query Execution (AQE) Internals 2 == page == Too many small tasks Scheduling overhead With AQE: → Spark merges small partitions automatically Fewer, optimal-sized tasks Result: Less overhead Better resource usage 2 Dynamic Join Strategy Switching At planning time, Spark may choose: Sort-Merge Join. But at runtime it may discover: One side is actually small enough to broadcast. AQE can switch: Sort-Merge → Broadcast Join Without you changing code. This can reduce shuffle dramatically. Skew Join Optimization Spark Adaptive Query Execution (AQE) Internals 3"
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"== page == If one partition is much larger: Without AQE: One executor overloaded → Long-running task Potential OOM With AQE: → Spark splits skewed partition → Processes it in parallel → Reduces imbalance This is huge for production workloads. How to Enable AQE spark.conf.set(""spark.sql.adaptive.enabled"", ""true"") Other useful configs: spark.conf.set(""spark.sql.adaptive.coalescePartitions.enabled"", ""true"") spark.conf.set(""spark.sql.adaptive.skewJoin.enabled"", ""true"") In Spark 3+, AQE is often enabled by default. Spark Adaptive Query Execution (AQE) Internals == page == How to Verify AQE is Working In Spark UI: Look at SQL tab Check ""AdaptiveSparkPlan"" in physical plan Compare initial plan vs final plan You'll see plan changes during execution. That's AQE in action. When AQE Doesn't Help AQE is powerful — but not magic. It won't fix: Poor partitioning strategy Massive data skew beyond threshold Bad UDF design Memory misconfiguration It optimizes around runtime statistics — not architectural flaws. Production Insight Spark Adaptive Query Execution (AQE) Internals 5 == page == Before AQE: Engineers manually tuned everything. After AQE: - Spark handles many optimizations automatically. But...Senior engineers still: Inspect execution plans Monitor skew behavior Validate join strategy Tune thresholds carefully Automation helps. Understanding still wins. Final Takeaway AQE makes Spark adaptive. But if you don't understand: How shuffle works How joins are selected How partition sizing affects execution You won't know whether AQE helped — or hid a deeper issue. Distributed systems are not about writing queries. They're about understanding execution. Spark Adaptive Query Execution (AQE) Internals 6 == page == AQE is Spark's way of becoming smarter at runtime. Your job is to be smarter than the runtime. Thank You.! Happy Learning... Spark Adaptive Query Execution (AQE) Internals"
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volum

In [0]:
from pyspark.sql import functions as F

# Add a unique, incremental id column before saving
df_chunks = df_chunks.withColumn("id", F.monotonically_increasing_id())


# Save the chunked data with id to the Delta table for retrieval and embedding
df_chunks.write.format("delta").mode("overwrite").option("mergeSchema", "true").saveAsTable(chunked_table)

display(spark.read.table(chunked_table))

path,chunk,id
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"Spark Adaptive Query Execution (AQE) Internals Prepared By: Kaviraj T U How Spark Rewrites Your Plan at Runtime Many engineers tune Spark jobs manually: Adjust shuffle partitions Change join strategy Handle skew manually Increase executor memory But modern Spark can do something powerful: It can change the execution plan at runtime. That feature is called Adaptive Query Execution (AQE). And understanding its internals is a senior-level Spark skill. What is AQE? Normally, Spark: Builds a logical plan 2 Optimizes it Creates a physical plan Executes it That plan is fixed before execution starts. Spark Adaptive Query Execution (AQE) Internals == page == Problem? Spark doesn't know the real/data distribution until shuffle happens. AQE solves this. It collects runtime statistics and modifies the physical plan dynamically. When Does AQE Kick In? AQE activates after a shuffle stage completes. At that moment Spark now knows: Actual partition sizes Real data distribution True join sizes Skewed partitions Using this information, it can re-optimize the next stage. What AQE Actually Optimizes Dynamic Shuffle Partition Coalescing Default shuffle partitions: spark.sql.shuffle.partitions=200 But what if data is small? Without AQE: → 200 tiny partitions Spark Adaptive Query Execution (AQE) Internals 2 == page == Too many small tasks Scheduling overhead With AQE: → Spark merges small partitions automatically Fewer, optimal-sized tasks Result: Less overhead Better resource usage 2 Dynamic Join Strategy Switching At planning time, Spark may choose: Sort-Merge Join. But at runtime it may discover: One side is actually small enough to broadcast. AQE can switch: Sort-Merge → Broadcast Join Without you changing code. This can reduce shuffle dramatically. Skew Join Optimization Spark Adaptive Query Execution (AQE) Internals 3",0
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"== page == If one partition is much larger: Without AQE: One executor overloaded → Long-running task Potential OOM With AQE: → Spark splits skewed partition → Processes it in parallel → Reduces imbalance This is huge for production workloads. How to Enable AQE spark.conf.set(""spark.sql.adaptive.enabled"", ""true"") Other useful configs: spark.conf.set(""spark.sql.adaptive.coalescePartitions.enabled"", ""true"") spark.conf.set(""spark.sql.adaptive.skewJoin.enabled"", ""true"") In Spark 3+, AQE is often enabled by default. Spark Adaptive Query Execution (AQE) Internals == page == How to Verify AQE is Working In Spark UI: Look at SQL tab Check ""AdaptiveSparkPlan"" in physical plan Compare initial plan vs final plan You'll see plan changes during execution. That's AQE in action. When AQE Doesn't Help AQE is powerful — but not magic. It won't fix: Poor partitioning strategy Massive data skew beyond threshold Bad UDF design Memory misconfiguration It optimizes around runtime statistics — not architectural flaws. Production Insight Spark Adaptive Query Execution (AQE) Internals 5 == page == Before AQE: Engineers manually tuned everything. After AQE: - Spark handles many optimizations automatically. But...Senior engineers still: Inspect execution plans Monitor skew behavior Validate join strategy Tune thresholds carefully Automation helps. Understanding still wins. Final Takeaway AQE makes Spark adaptive. But if you don't understand: How shuffle works How joins are selected How partition sizing affects execution You won't know whether AQE helped — or hid a deeper issue. Distributed systems are not about writing queries. They're about understanding execution. Spark Adaptive Query Execution (AQE) Internals 6 == page == AQE is Spark's way of becoming smarter at runtime. Your job is to be smarter than the runtime. Thank You.! Happy Learning... Spark Adaptive Query Execution (AQE) Internals",1
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_gena

In [0]:
%sql
select * from dmi_genomics_qa_team_consenting_practice.bg_genai.docs_chunked

path,chunk,id
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"Spark Adaptive Query Execution (AQE) Internals Prepared By: Kaviraj T U How Spark Rewrites Your Plan at Runtime Many engineers tune Spark jobs manually: Adjust shuffle partitions Change join strategy Handle skew manually Increase executor memory But modern Spark can do something powerful: It can change the execution plan at runtime. That feature is called Adaptive Query Execution (AQE). And understanding its internals is a senior-level Spark skill. What is AQE? Normally, Spark: Builds a logical plan 2 Optimizes it Creates a physical plan Executes it That plan is fixed before execution starts. Spark Adaptive Query Execution (AQE) Internals == page == Problem? Spark doesn't know the real/data distribution until shuffle happens. AQE solves this. It collects runtime statistics and modifies the physical plan dynamically. When Does AQE Kick In? AQE activates after a shuffle stage completes. At that moment Spark now knows: Actual partition sizes Real data distribution True join sizes Skewed partitions Using this information, it can re-optimize the next stage. What AQE Actually Optimizes Dynamic Shuffle Partition Coalescing Default shuffle partitions: spark.sql.shuffle.partitions=200 But what if data is small? Without AQE: → 200 tiny partitions Spark Adaptive Query Execution (AQE) Internals 2 == page == Too many small tasks Scheduling overhead With AQE: → Spark merges small partitions automatically Fewer, optimal-sized tasks Result: Less overhead Better resource usage 2 Dynamic Join Strategy Switching At planning time, Spark may choose: Sort-Merge Join. But at runtime it may discover: One side is actually small enough to broadcast. AQE can switch: Sort-Merge → Broadcast Join Without you changing code. This can reduce shuffle dramatically. Skew Join Optimization Spark Adaptive Query Execution (AQE) Internals 3",0
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_genai/volume/spark/AQE.pdf,"== page == If one partition is much larger: Without AQE: One executor overloaded → Long-running task Potential OOM With AQE: → Spark splits skewed partition → Processes it in parallel → Reduces imbalance This is huge for production workloads. How to Enable AQE spark.conf.set(""spark.sql.adaptive.enabled"", ""true"") Other useful configs: spark.conf.set(""spark.sql.adaptive.coalescePartitions.enabled"", ""true"") spark.conf.set(""spark.sql.adaptive.skewJoin.enabled"", ""true"") In Spark 3+, AQE is often enabled by default. Spark Adaptive Query Execution (AQE) Internals == page == How to Verify AQE is Working In Spark UI: Look at SQL tab Check ""AdaptiveSparkPlan"" in physical plan Compare initial plan vs final plan You'll see plan changes during execution. That's AQE in action. When AQE Doesn't Help AQE is powerful — but not magic. It won't fix: Poor partitioning strategy Massive data skew beyond threshold Bad UDF design Memory misconfiguration It optimizes around runtime statistics — not architectural flaws. Production Insight Spark Adaptive Query Execution (AQE) Internals 5 == page == Before AQE: Engineers manually tuned everything. After AQE: - Spark handles many optimizations automatically. But...Senior engineers still: Inspect execution plans Monitor skew behavior Validate join strategy Tune thresholds carefully Automation helps. Understanding still wins. Final Takeaway AQE makes Spark adaptive. But if you don't understand: How shuffle works How joins are selected How partition sizing affects execution You won't know whether AQE helped — or hid a deeper issue. Distributed systems are not about writing queries. They're about understanding execution. Spark Adaptive Query Execution (AQE) Internals 6 == page == AQE is Spark's way of becoming smarter at runtime. Your job is to be smarter than the runtime. Thank You.! Happy Learning... Spark Adaptive Query Execution (AQE) Internals",1
dbfs:/Volumes/dmi_genomics_qa_team_consenting_practice/bg_gena